# Kafka Demo — Lab 3

### Connect to Kafka Broker Server
Open an SSH tunnel in your terminal and leave it running while you use this notebook.
Replace `<NetID>` with your UIC NetID:
```
ssh -o ServerAliveInterval=60 -L 9092:localhost:9092 <NetID>@cs544-f26.cs.uic.edu -NTf
```

### To kill connection
```
lsof -ti:9092 | xargs kill -9
```

### Setup
```
python -m pip install kafka-python
```

See [bug_list.md](./bug_list.md) for frequent bugs and solutions.

In [ ]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer

# Update this for your own recitation section :)
topic = 'recitation-sina'

### Producer Mode -> Writes Data to Broker

In [7]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

producer = KafkaProducer(
    bootstrap_servers=["localhost:9092"],
    value_serializer=lambda x: dumps(x).encode("utf-8")
)

cities = ["Chicago", "Boston", "Tehran"]

# Write data via the producer
print("Writing to Kafka Broker")
for i in range(10):
    data = f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")},{cities[randint(0,len(cities)-1)]},{randint(18, 32)}ºC'
    print(f"Writing: {data}")
    producer.send(topic=topic, value=data)
    sleep(1)

Writing to Kafka Broker
Writing: 2026-10-07 11:28:12,Chicago,19ºC
Writing: 2026-10-07 11:28:14,Chicago,27ºC
Writing: 2026-10-07 11:28:15,Boston,31ºC
Writing: 2026-10-07 11:28:16,Chicago,24ºC
Writing: 2026-10-07 11:28:17,Chicago,27ºC
Writing: 2026-10-07 11:28:18,Boston,21ºC
Writing: 2026-10-07 11:28:19,Chicago,30ºC
Writing: 2026-10-07 11:28:20,Chicago,22ºC
Writing: 2026-10-07 11:28:21,Chicago,31ºC
Writing: 2026-10-07 11:28:22,Chicago,19ºC


### Consumer Mode -> Reads Data from Broker

In [ ]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html


consumer = KafkaConsumer(
    topic,
    bootstrap_servers=["localhost:9092"],
    auto_offset_reset="earliest",
    group_id="lab3-sina",
    enable_auto_commit=True,
    auto_commit_interval_ms=1000,
    consumer_timeout_ms=15000
)

print("Reading Kafka Broker")

try:
    with open("kafka_log.csv", "a", encoding="utf-8") as log_file:
        for message in consumer:
            text = loads(message.value.decode("utf-8"))

            print(
                f"Partition={message.partition} "
                f"Offset={message.offset} "
                f"Message={text}"
            )

            log_file.write(text + "\n")
finally:
    consumer.close()

Reading Kafka Broker
Partition=0 Offset=10 Message=2026-10-07 11:28:12,Chicago,19ºC
Partition=0 Offset=11 Message=2026-10-07 11:28:14,Chicago,27ºC
Partition=0 Offset=12 Message=2026-10-07 11:28:15,Boston,31ºC
Partition=0 Offset=13 Message=2026-10-07 11:28:16,Chicago,24ºC
Partition=0 Offset=14 Message=2026-10-07 11:28:17,Chicago,27ºC
Partition=0 Offset=15 Message=2026-10-07 11:28:18,Boston,21ºC
Partition=0 Offset=16 Message=2026-10-07 11:28:19,Chicago,30ºC
Partition=0 Offset=17 Message=2026-10-07 11:28:20,Chicago,22ºC
Partition=0 Offset=18 Message=2026-10-07 11:28:21,Chicago,31ºC
Partition=0 Offset=19 Message=2026-10-07 11:28:22,Chicago,19ºC


# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [ ]:
#kcat command: connect to local Kafka broker, specify a topic, and consume messages from the earliest offset